# Phase 6 feature-ablation workflow

This is the single starting point for Phase 6. It is a read-only guide: it checks which local evidence packages exist and explains the next experiment. The model-fitting notebooks remain separate because each one creates a new evidence folder and refuses to overwrite it.

The original pre-recovery Phase 6.1/6.2 notebook content was replaced because it depended on generated Phase 3–5 reports that are unavailable in this checkout. The recovery workflow uses the rebuilt market-only dataset and protocol under `phase6_recovery_v1`.

## Workflow and notebook roles

1. Read the local status below. It does not fit a model or overwrite evidence.
2. `phase6_recovery_validation.ipynb` is retained as a completed Step 6.3 record. Run it only when rebuilding the Phase 6 recovery evidence from scratch.
3. `phase6_momentum_ablation.ipynb` is the completed Step 6.4 experiment. Do not rerun it into the same output folder.
4. `phase6_volatility_ablation.ipynb` is the completed Step 6.5 experiment. Do not rerun it into the same output folder.
5. Step 6.6 needs a new downside/trend notebook only after the Step 6.4 evidence discrepancy is reconciled.

Reusable fitting and leakage checks live in `src/model_training/feature_ablation/`.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

BRANCH = 'feature/model-evaluation'
COLAB_REPOSITORY = Path('/content/HalalifyAI')
if 'google.colab' in sys.modules:
    if not (COLAB_REPOSITORY / '.git').is_dir():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/frager098/HalalifyAI.git', str(COLAB_REPOSITORY)], check=True)
    root = COLAB_REPOSITORY.resolve()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt'), '-r', str(root / 'requirements-cleaning.txt')], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'src' / 'model_training').is_dir():
        root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.notebook_environment import ensure_phase6_recovery_inputs, prepare_repository
root = prepare_repository(branch=BRANCH, colab_repository=COLAB_REPOSITORY)
ensure_phase6_recovery_inputs(root)
%cd {root}
evidence_root = root / 'reports' / 'model_evaluation' / 'phase6_recovery_v1'
evidence_root

WindowsPath('D:/Arham/FYP/HalalifyAI/reports/model_evaluation/phase6_recovery_v1')

## Read completed local evidence

This cell only reads already generated JSON. It is safe to run repeatedly.

In [2]:
summary_paths = {
    'runner validation (6.3)': evidence_root / 'runner_validation.json',
    'momentum comparison (6.4)': evidence_root / 'momentum_ablation' / 'momentum_ablation_summary.json',
    'volatility comparison (6.5)': evidence_root / 'volatility_ablation' / 'volatility_ablation_summary.json',
}

for label, path in summary_paths.items():
    print(f'\n{label}:')
    if not path.exists():
        print(f'  Missing: {path.relative_to(root)}')
        continue
    report = json.loads(path.read_text(encoding='utf-8'))
    print('  status:', report['status'])
    for task, result in report.get('task_summaries', {}).items():
        print(f"  {task}: {result['decision']} ({result['primary_metric']} difference {result['mean_difference']:.5f})")


runner validation (6.3):
  status: validated

momentum comparison (6.4):
  status: complete
  return: inconclusive (mean_daily_spearman difference -0.00499)
  risk: inconclusive (macro_f1 difference 0.00022)

volatility comparison (6.5):
  status: complete
  return: inconclusive (mean_daily_spearman difference -0.00349)
  risk: retain (macro_f1 difference 0.01443)


## Before Step 6.6

The local volatility evidence completes Step 6.5: volatility is inconclusive for return and retained for risk. However, the regenerated local Step 6.4 momentum summary differs from the committed Step 6.4 record. Compare the saved hashes, Python/package versions, and runner code hash before accepting either momentum conclusion or beginning the next feature group.

When that reconciliation is recorded, Step 6.6 will test the downside/trend group with the same four chronological folds and matched feature-excluded reference.